In [148]:
categorical_cols = [
    'visit_type',
    'specialty_group',
    'reminder_channel',
    'sex_at_birth',
    'race_ethnicity_group',
    'preferred_language',
    'facility_type',
    'urban_rural_group',
    'payer_group',
    'service_line_focus',
    'weather_severity_group',
    'facility_name'
]
numerical_cols = [
    'appointment_weekday',
    'lead_days',
    'reminder_sent_flag',
    'reminder_days_before',
    'same_day_flag',
    'telehealth_flag',
    'distance_to_clinic_miles',
    'transportation_barrier_flag',
    'interpreter_needed_flag_x',
    'patient_age',
    'sdoh_risk_score_x',
    'portal_active_flag',
    'deprivation_quintile',
    'prior_completed_visits_12m',
    'prior_no_show_count_12m',
    'prior_cancel_count_12m',
    'chronic_condition_count',
    'interpreter_needed_flag_y',
    'sdoh_risk_score_y',
    'teaching_site_flag',
    'quarter',
    'week_of_year',
    'weekend_flag',
    'holiday_flag',
    'flu_season_flag',
]

In [149]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score




merged_data = pd.read_csv('../data/clean/merged_data_cleaned.csv')
target = merged_data['no_show_flag']
features = merged_data.drop(columns=[
    'no_show_flag', 'appointment_id', 'patient_id', 'provider_id',
    'facility_id', 'pcp_provider_id', 'zip_area_id_x', 'zip_area_id_y', 
    'zip_area_id', 'month_num', 'weekday_num','payer_id','payer_primary_id', 'wait_to_room_minutes'
])

x_train, x_test, y_train, y_test = train_test_split( features, target, test_size=0.2, random_state=42, stratify=target)

scaler = StandardScaler()
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

train_encoded_df = encoder.fit_transform(x_train[categorical_cols])
test_encoded_df = encoder.transform(x_test[categorical_cols])

x_train[numerical_cols] = scaler.fit_transform(x_train[numerical_cols])
x_test[numerical_cols] = scaler.transform(x_test[numerical_cols])

x_train = x_train.drop(columns=categorical_cols)
x_test = x_test.drop(columns=categorical_cols)

x_train = x_train.join(train_encoded_df)
x_test = x_test.join(test_encoded_df)

# type(train_encoded_df)
# type(x_train)



In [150]:

baseline = DummyClassifier(strategy='most_frequent') # accuracy = 86
# baseline = DummyClassifier(strategy='stratified', random_state=42) accuracy = 76

baseline.fit(x_train, y_train)
baseline_predictions = baseline.predict(x_test)


In [151]:
accuracy = accuracy_score(y_test, baseline_predictions)
print("Baseline Accuracy:", accuracy)

Baseline Accuracy: 0.8634722222222222


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.model_selection import GridSearchCV

decision_tree_algo = DecisionTreeClassifier(max_depth=2,random_state=42)

param_grid = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 10, 20, 50],
    'min_samples_leaf': [1, 5, 10, 20],
    'criterion': ['gini', 'entropy']
}
decision_tree = DecisionTreeClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=decision_tree,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(x_train, y_train)

# overfitting

Fitting 5 folds for each of 192 candidates, totalling 960 fits


In [ ]:
best_Hyperparameters_decision_tree = grid_search.best_estimator_

decision_tree_predictions = best_Hyperparameters_decision_tree.predict(x_test)

accuracy = accuracy_score(y_test, decision_tree_predictions)
print(accuracy)
con_matrix = confusion_matrix(y_test,decision_tree_predictions)
print(con_matrix)
report = classification_report(y_test,decision_tree_predictions)
print(report)

1.0
[[6217    0]
 [   0  983]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6217
           1       1.00      1.00      1.00       983

    accuracy                           1.00      7200
   macro avg       1.00      1.00      1.00      7200
weighted avg       1.00      1.00      1.00      7200

